# Milestone 2: the recogniser, alignment and clips

**Goal:** cut every chapter into short clips whose text matches the audio
exactly, and measure how long each character lasts, using a speech recogniser
built from scratch on the same data. See `PLAN.md`, section 4, milestone 2.

The recogniser is trained three times, each round on better data:

| Round | Clips come from | Model | Used for |
|---|---|---|---|
| 0 | constant-reading-rate guess, cut at pauses, ~25 s | stride 2 (half frame rate) | first chapter alignment |
| 1 | round-0 alignment, cut at sentence ends, 2–12 s | stride 1 (full frame rate) | final alignment and durations |
| 2 (only if needed) | round-1 alignment | stride 1 | checking the alignment has settled |

**Runtime:** GPU (Runtime → Change runtime type → T4 GPU). Needs notebook 01's
archive on Drive. Each training run saves checkpoints to Drive; if Colab
disconnects, rerun the setup cells and the training cell, and it continues.

**Gate to milestone 3:**
- listen to 50 random clips while reading their text: at least 95% match exactly;
- character error rate (CER) on dev clips, greedy decoding: 10% or lower;
- duration plots for ten clips: character boundaries sit roughly where the sound changes.

**Background reading:** Hannun, "Sequence Modeling with CTC" (Distill, 2017);
Park et al. 2019, SpecAugment; Kürzinger et al. 2020, CTC segmentation.

## Setup

In [ ]:
# Get the code, install it, and connect Google Drive.
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/nicholas-on/TTS.git"
BRANCH = "main"
CODE_DIR = Path("/content/TTS")

def github_token():
    """The repository is private, so cloning needs a GitHub token, stored as a
    Colab secret named GITHUB_TOKEN with "Notebook access" switched on.
    See the README, "Running the proof of concept on Google Colab"."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except ImportError:
        return None  # not in Colab
    except Exception as e:  # secret missing, or notebook access not granted
        print(f"No GITHUB_TOKEN available ({type(e).__name__}).")
        return None

def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(
            f"git {args[0]} failed:\n{message}\n"
            "If this says 'Repository not found' or 'could not read Username', the token is "
            "missing or wrong: add a Colab secret named GITHUB_TOKEN (key icon, left sidebar), "
            "switch on 'Notebook access', and rerun this cell.")

token = github_token()
if not CODE_DIR.exists():
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    git("clone", "-q", "-b", BRANCH, url, str(CODE_DIR))
else:
    git("-C", str(CODE_DIR), "pull", "-q")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE_DIR)], check=True)
if str(CODE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "src"))
# If this cell is rerun in the same session after new code was pulled, forget
# the old version so the imports below load the new one.
for name in [m for m in sys.modules if m == "bibletts" or m.startswith("bibletts.")]:
    del sys.modules[name]

from bibletts.colab import mount_drive, gpu_name, disk_free_gb
from bibletts.config import load_config

mount_drive()
cfg = load_config(CODE_DIR / "configs" / "data.yaml")
DRIVE = Path(cfg["paths"]["drive_root"])   # durable
LOCAL = Path(cfg["paths"]["local_root"])   # fast, wiped when the session ends
RAW = DRIVE / "raw"                        # downloaded originals
for d in (DRIVE, LOCAL, RAW):
    d.mkdir(parents=True, exist_ok=True)
print("GPU:", gpu_name(), "| free local disk:", round(disk_free_gb()), "GB")

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from bibletts.audio import MelConfig, MelSpectrogram, load_audio
from bibletts.corpus import read_manifest, load_chapter, restore_archive
from bibletts.text import CTCVocab, TTSVocab
from bibletts.ctc.data import compute_chapter_mels, load_mels, mel_stats
from bibletts.ctc.pipeline import (save_segments, load_segments, make_rough_segments,
                                   align_chapters, make_aligned_segments, add_durations)
from bibletts.ctc.train import train_round, load_model, read_log, evaluate
from bibletts.ctc.segment import cut_shifts
from bibletts.ctc.data import ClipDataset

ctc_cfg = load_config(CODE_DIR / "configs" / "ctc.yaml")
CTC_DIR = DRIVE / "ctc"           # models and clip lists, kept on Drive
CTC_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: no GPU. Training will be very slow. Runtime -> Change runtime type -> T4 GPU.")

# The prepared corpus from notebook 01.
if not (LOCAL / "manifest.jsonl").exists():
    restore_archive(DRIVE / "prepared" / "chapters_v1.tar", LOCAL.parent)
manifest = read_manifest(LOCAL)
metas = [load_chapter(LOCAL, r["book"], r["chapter"]) for r in manifest]
by_split = {}
for m in metas:
    by_split.setdefault(m["split"], []).append(m)
print({s: len(v) for s, v in by_split.items()})

mel_cfg = MelConfig.from_dict(cfg["audio"])
hop_s = mel_cfg.hop_length / mel_cfg.sample_rate
ctc_vocab = CTCVocab(cfg["text"]["alphabet"])
tts_vocab = TTSVocab(cfg["text"]["alphabet"])

def play(seg, label=True):
    """Play one clip and show its text."""
    wav = LOCAL / "chapters" / seg["chapter"] / "audio.wav"
    audio = load_audio(wav, mel_cfg.sample_rate, seg["start_s"], seg["end_s"] - seg["start_s"])
    if label:
        print(f"{seg['id']} ({seg['end_s'] - seg['start_s']:.1f} s): {seg['text']}")
    display(Audio(audio, rate=mel_cfg.sample_rate))

## 1. Mel-spectrograms

Every chapter's mel-spectrogram is computed once and kept on the local disk
(about 1 GB as 16-bit floats). Clips are slices of these. The recogniser
normalises each mel band with the mean and spread measured on the training
chapters.

In [ ]:
mel = MelSpectrogram(mel_cfg)
compute_chapter_mels(LOCAL, [m["id"] for m in metas], mel, device=device)
mels = load_mels(LOCAL, [m["id"] for m in metas])
mel_mean, mel_std = mel_stats({m["id"]: mels[m["id"]] for m in by_split["train"]})
print(f"{sum(x.shape[1] for x in mels.values()) * hop_s / 3600:.1f} h of mel frames loaded")

## 2. Round 0: rough clips, no model

Each chapter is cut every ~25 seconds at a detected pause near where a verse
*should* end if the reader kept a constant pace. Listen to a few: the start
and end of a clip may include a word or two from the neighbouring verse. That
is expected. CTC training tolerates a small fraction of wrong labels, and the
next round fixes the edges.

In [ ]:
round0 = make_rough_segments(by_split["train"] + by_split["dev"], ctc_cfg["segment"], hop_s)
save_segments(round0, CTC_DIR / "round0_segments.jsonl")
r0_train = [s for s in round0 if s["split"] == "train"]
r0_dev = [s for s in round0 if s["split"] == "dev"]
lengths = np.array([s["end_s"] - s["start_s"] for s in round0])
print(f"{len(r0_train)} train clips, {len(r0_dev)} dev clips; "
      f"length median {np.median(lengths):.1f} s, min {lengths.min():.1f}, max {lengths.max():.1f}; "
      f"{sum(s['end_s'] - s['start_s'] for s in r0_train) / 3600:.2f} h of train audio")

rng = np.random.default_rng(0)
for i in rng.choice(len(r0_train), 3, replace=False):
    play(r0_train[i])

## 3. Train round 0

About 30 minutes on a T4 (a guess; the first log lines give the real speed).
The loss should fall steadily. Dev CER is printed every 500 steps along with
two example transcriptions.

**If it does not learn** (loss flat after a few epochs, dev CER stuck near
100%): the rough clips are too noisy. The fallback in `PLAN.md` is to mark
verse boundaries by hand for about 30 minutes of audio (Audacity's label track
works) and train on that instead.

In [ ]:
rc = ctc_cfg["rounds"]["round0"]
model0 = train_round(r0_train, r0_dev, mels, ctc_vocab, ctc_cfg, stride=rc["stride"],
                     epochs=rc["epochs"], out_dir=CTC_DIR / "round0", mel_mean=mel_mean,
                     mel_std=mel_std, hop_s=hop_s, device=device)

In [ ]:
def plot_log(out_dir, title):
    log = read_log(out_dir)
    loss = [(r["step"], r["loss"]) for r in log if "loss" in r]
    dev = [(r["step"], r["dev_cer"]) for r in log if "dev_cer" in r]
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    if loss: axes[0].plot(*zip(*loss)); axes[0].set(title=f"{title}: train CTC loss", xlabel="step")
    if dev: axes[1].plot(*zip(*dev), marker="o"); axes[1].set(title=f"{title}: dev CER", xlabel="step", ylim=(0, 1))
    plt.show()
    final = [r["final_dev_cer"] for r in log if "final_dev_cer" in r]
    if final: print(f"Final dev CER: {final[-1]:.3f}")

plot_log(CTC_DIR / "round0", "round 0")

## 4. Round 1: align whole chapters, cut exact clips

The round-0 model gives, for every frame of a chapter, the probability of each
character. Forced alignment (Viterbi) then finds the single most likely timing
of the chapter's known text. Every character gets a time, so clips can be cut
exactly at sentence or verse ends that fall in a pause.

Each clip is scored by the mean log-probability of its aligned path; the
lowest-scoring few percent are dropped. Low scores mean the audio and text
disagree there: the reader skipped or added words, or there is music or noise.

All splits are aligned, so dev and test chapters also get exact clips. Their
audio is never used for training.

In [ ]:
def align_and_cut(model, stride, name):
    alignments, failures = align_chapters(model, metas, mels, ctc_vocab, ctc_cfg["align"],
                                          hop_s, device)
    if failures:
        print("Alignment failed for:", failures)
    kept, dropped = make_aligned_segments(metas, alignments, ctc_cfg["segment"], hop_s, stride)
    save_segments(kept, CTC_DIR / f"{name}_segments.jsonl")
    save_segments(dropped, CTC_DIR / f"{name}_dropped.jsonl")
    for split in by_split:
        group = [s for s in kept if s["split"] == split]
        hours = sum(s["end_s"] - s["start_s"] for s in group) / 3600
        audio_h = sum(m["duration_s"] for m in by_split[split]) / 3600
        print(f"{split:>11}: {len(group):5d} clips, {hours:5.2f} h of {audio_h:5.2f} h chapter audio")
    return alignments, kept, dropped

alignments1, round1, dropped1 = align_and_cut(model0, ctc_cfg["rounds"]["round0"]["stride"], "round1")

In [ ]:
print("Worst dropped clips (listen: what went wrong?)")
for s in sorted(dropped1, key=lambda s: s["score"])[:3]:
    print(f"score {s['score']:.2f}"); play(s)
print("\nRandom kept clips")
r1_train = [s for s in round1 if s["split"] == "train"]
for i in rng.choice(len(r1_train), 4, replace=False):
    play(r1_train[i])

## 5. Train round 1 at full frame rate

A fresh model on the exact clips. This one runs at the mel frame rate (stride
1), so its alignment can give each character a duration in mel frames, which
is what the TTS model needs. Expect a clearly lower dev CER than round 0.

In [ ]:
rc = ctc_cfg["rounds"]["round1"]
r1_dev = [s for s in round1 if s["split"] == "dev"]
model1 = train_round(r1_train, r1_dev, mels, ctc_vocab, ctc_cfg, stride=rc["stride"],
                     epochs=rc["epochs"], out_dir=CTC_DIR / "round1", mel_mean=mel_mean,
                     mel_std=mel_std, hop_s=hop_s, device=device)
plot_log(CTC_DIR / "round1", "round 1")

## 6. Realign and check the alignment has settled

Align again with the better model and compare where clips end. If nearly all
clip ends moved by less than 50 ms, the alignment has settled and these are the
final clips. If not, run the optional section 6b, which trains once more.

In [ ]:
alignments2, round2, dropped2 = align_and_cut(model1, ctc_cfg["rounds"]["round1"]["stride"], "round2")
shifts = cut_shifts(round1, round2)
print(f"Clip ends within 50 ms of round 1: {np.mean(shifts < 0.05):.1%} "
      f"(median shift {1000 * np.median(shifts):.0f} ms)")
FINAL_MODEL, final_alignments, final_clips = model1, alignments2, round2

### 6b. Optional: one more round

Only if fewer than about 90% of clip ends were within 50 ms above.

In [ ]:
RUN_ROUND_2 = False
if RUN_ROUND_2:
    rc = ctc_cfg["rounds"]["round2"]
    model2 = train_round([s for s in round2 if s["split"] == "train"],
                         [s for s in round2 if s["split"] == "dev"], mels, ctc_vocab, ctc_cfg,
                         stride=rc["stride"], epochs=rc["epochs"], out_dir=CTC_DIR / "round2",
                         mel_mean=mel_mean, mel_std=mel_std, hop_s=hop_s, device=device)
    alignments3, round3, dropped3 = align_and_cut(model2, rc["stride"], "round3")
    shifts = cut_shifts(round2, round3)
    print(f"Clip ends within 50 ms of round 2: {np.mean(shifts < 0.05):.1%}")
    FINAL_MODEL, final_alignments, final_clips = model2, alignments3, round3

## 7. Durations for the TTS model

Each clip gets its TTS tokens (letters, typed word boundaries, start and end
tokens) and the number of mel frames each token lasts. Convention: a character
owns the frames from its own CTC spike up to the next character's spike; the
start token owns the leading silence and the end token the trailing silence.
The durations of a clip always add up to its number of mel frames.

In [ ]:
clips, failed = add_durations(final_clips, final_alignments, tts_vocab, hop_s,
                              stride=FINAL_MODEL.stride)
print(f"{len(clips)} clips with durations; {len(failed)} failed: {failed[:10]}")
save_segments(clips, CTC_DIR / "clips_final.jsonl")
print("Saved", CTC_DIR / "clips_final.jsonl")

In [ ]:
from bibletts.viz import plot_durations

def show_durations(clip):
    m = mels[clip["chapter"]][:, clip["mel_start"]: clip["mel_end"]].astype(np.float32)
    labels = [tts_vocab.tokens[t].strip() or "_" for t in clip["tokens"]]
    labels[0], labels[-1] = "<s>", "</s>"
    plot_durations(m, labels, clip["durations"], mel_cfg.sample_rate, mel_cfg.hop_length,
                   title=clip["text"])
    plt.show()
    play(clip, label=False)

for i in rng.choice(len(clips), 10, replace=False):
    show_durations(clips[i])

## 8. Gate checks

### Listening test

Fifty random training clips, ten per run of the cell. For each, read the text
while listening. Count clips where any word is missing, extra or different, or
the audio is cut mid-word. Change `PAGE` from 0 to 4.

In [ ]:
listen = [clips[i] for i in np.random.default_rng(42).choice(len(clips), 50, replace=False)]
PAGE = 0
for s in listen[PAGE * 10:(PAGE + 1) * 10]:
    play(s)

**Result:** __ of 50 clips matched exactly (gate: at least 48).

### Character error rate

Dev and test clips come from chapters never used for training. Greedy
decoding, no language model. The test numbers are the floor for milestone 5:
this is how well the recogniser reads real speech.

In [ ]:
for split in ["dev", "test_same", "test_style", "test_names"]:
    group = [s for s in clips if s["split"] == split]
    if group:
        result = evaluate(FINAL_MODEL, ClipDataset(group, mels, ctc_vocab), device, max_clips=10**9)
        print(f"{split:>11}: CER {result['cer']:.3f} on {len(group)} clips")
        for ref, hyp in result["examples"][:1]:
            print(f"   ref: {ref}\n   hyp: {hyp}")

### Checklist

- [ ] At least 48 of 50 clips match their text exactly
- [ ] Dev CER at most 0.10
- [ ] Duration plots: boundaries roughly where the sound changes
- [ ] `ctc/clips_final.jsonl` and the final recogniser (`ctc/round1/final.pt`, or
      `round2`) are on Drive
- [ ] Note the dev and test CERs here: dev __, test_same __, test_style __, test_names __